# Hotel Management AI Agent

In [127]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio 
import ollama
import sqlite3 
import re
from datetime import date
from pathlib import Path

In [128]:
load_dotenv(override=True)

PROVIDER = "groq"   # change to "ollama" to use the local Llama model instead

if PROVIDER == "ollama":
    client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    MODEL = "llama3.1:8b"
elif PROVIDER == "groq":
    client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=os.getenv("GROQ_API_KEY"))
    MODEL = "openai/gpt-oss-120b"

DB = str(Path.cwd() / "hotel.db")

# Database

In [129]:
def create_customers_table():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS customers (
                customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
                full_name TEXT NOT NULL,
                phone TEXT NOT NULL,
                email TEXT,
                room_number TEXT NOT NULL,
                check_in TEXT NOT NULL,
                check_out TEXT NOT NULL,
                special_requests TEXT
            )
        """)
        conn.commit()

create_customers_table()

In [130]:
# check the table is exists
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    print(cursor.fetchall())

[('customers',), ('sqlite_sequence',)]


# Tool functions (validation, save, view, update, delete)

Validation (validate the customer data)

In [131]:
def validate_customer_data(
    full_name=None, phone=None, email=None,
    room_number=None, check_in=None, check_out=None,
    require_all=False
):
    fields = {
        "Full name": full_name,
        "Phone number": phone,
        "Room number": room_number,
        "Check-in date": check_in,
        "Check-out date": check_out,
    }

    if require_all:
        for label, value in fields.items():
            if not isinstance(value, str) or not value.strip():
                return False, f"{label} is required."

    for label, value in fields.items():
        if value is not None and not isinstance(value, str):
            return False, f"{label} must be text."

    if full_name is not None and not full_name.strip():
        return False, "Full name cannot be empty."

    if phone is not None and not re.fullmatch(r"\+?\d{9,15}", phone.strip()):
        return False, "Phone number must be 9 to 15 digits, optionally starting with +."

    if room_number is not None and not room_number.strip():
        return False, "Room number cannot be empty."

    if email is not None:
        if not isinstance(email, str):
            return False, "Email address must be text."
        if email.strip() and not re.fullmatch(
            r"[^@\s]+@[^@\s]+\.[^@\s]+", email.strip()
        ):
            return False, "Email address doesn't look valid."

    parsed_dates = {}
    for label, value in (
        ("Check-in", check_in),
        ("Check-out", check_out),
    ):
        if value is not None:
            if not isinstance(value, str) or not re.fullmatch(
                r"\d{4}-\d{2}-\d{2}", value.strip()
            ):
                return False, f"{label} date must use YYYY-MM-DD format."
            try:
                parsed_dates[label] = date.fromisoformat(value.strip())
            except ValueError:
                return False, f"{label} date is not a valid calendar date."

    if (
        "Check-in" in parsed_dates
        and "Check-out" in parsed_dates
        and parsed_dates["Check-out"] <= parsed_dates["Check-in"]
    ):
        return False, "Check-out date must be after the check-in date."

    return True, None

Save customer data

In [132]:
def save_customer(full_name, phone, room_number, check_in, check_out,
                   email=None, special_requests=None):
    valid, error = validate_customer_data(
        full_name=full_name, phone=phone, email=email,
        room_number=room_number, check_in=check_in, check_out=check_out, require_all=True
    )
    if not valid:
        return {"success": False, "error": error}

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO customers (full_name, phone, email, room_number,
                                    check_in, check_out, special_requests)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (full_name, phone, email, room_number, check_in, check_out, special_requests))
        conn.commit()
        new_id = cursor.lastrowid

    return {"success": True, "customer_id": new_id}

In [133]:
#Save a customer to test the function
#print(save_customer("Amara Perera", "+94712223344", "A12", "2026-11-01", "2026-11-03", email="amara@example.com"))

In [134]:
# Invalid input, rejected
print(save_customer("Bad Phone", "123", "101", "2026-10-01", "2026-10-04"))
# Expect: success False, error about phone digits

print(save_customer("Bad Dates", "0712345678", "101", "2026-10-05", "2026-10-01"))
# Expect: success False, error about check-out after check-in

{'success': False, 'error': 'Phone number must be 9 to 15 digits, optionally starting with +.'}
{'success': False, 'error': 'Check-out date must be after the check-in date.'}


View (Search customer data)

In [135]:
def view_customer(customer_id=None, name=None):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        if customer_id is not None:
            cursor.execute("SELECT * FROM customers WHERE customer_id = ?", (customer_id,))
        elif name is not None:
            cursor.execute("SELECT * FROM customers WHERE full_name LIKE ?", (f"%{name}%",))
        else:
            return {"success": False, "error": "Provide a customer_id or a name to search."}

        columns = [d[0] for d in cursor.description]
        rows = cursor.fetchall()

    if not rows:
        return {"success": False, "error": "No matching customer found."}

    results = [dict(zip(columns, row)) for row in rows]

    if len(results) > 1:
        return {"success": True, "multiple_matches": True, "results": results}

    return {"success": True, "multiple_matches": False, "result": results[0]}

In [136]:
# view by id
print(view_customer(customer_id=1))

{'success': True, 'multiple_matches': False, 'result': {'customer_id': 1, 'full_name': 'Amara Perera', 'phone': '+94712223344', 'email': 'amara@example.com', 'room_number': '118', 'check_in': '2026-11-01', 'check_out': '2026-11-03', 'special_requests': None}}


In [137]:
# view by name
print(view_customer(name="Amara Perera"))

{'success': True, 'multiple_matches': False, 'result': {'customer_id': 1, 'full_name': 'Amara Perera', 'phone': '+94712223344', 'email': 'amara@example.com', 'room_number': '118', 'check_in': '2026-11-01', 'check_out': '2026-11-03', 'special_requests': None}}


In [138]:
# View, not found
print(view_customer(customer_id=99))

{'success': False, 'error': 'No matching customer found.'}


Update 

In [139]:
def update_customer(customer_id, full_name=None, phone=None, email=None,
                     room_number=None, check_in=None, check_out=None,
                     special_requests=None):
    existing = view_customer(customer_id=customer_id)
    if not existing["success"]:
        return {"success": False, "error": "Customer not found."}

    current = existing["result"]

    # Validate against the merged (existing + new) values so cross-field
    # rules like check_out > check_in still hold
    merged_check_in = check_in if check_in is not None else current["check_in"]
    merged_check_out = check_out if check_out is not None else current["check_out"]

    valid, error = validate_customer_data(
        full_name=full_name, phone=phone, email=email,
        room_number=room_number, check_in=merged_check_in, check_out=merged_check_out
    )
    if not valid:
        return {"success": False, "error": error}

    fields = {
        "full_name": full_name, "phone": phone, "email": email,
        "room_number": room_number, "check_in": check_in,
        "check_out": check_out, "special_requests": special_requests
    }
    changes = {k: v for k, v in fields.items() if v is not None}

    if not changes:
        return {"success": False, "error": "No fields provided to update."}

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        set_clause = ", ".join(f"{k} = ?" for k in changes)
        cursor.execute(
            f"UPDATE customers SET {set_clause} WHERE customer_id = ?",
            (*changes.values(), customer_id)
        )
        conn.commit()

    return {"success": True, "customer_id": customer_id, "changed_fields": changes}

In [140]:
# update customer data
#print(update_customer(customer_id=1, room_number="300"))
# Expect: success True, changed_fields: {'room_number': '300'}
#print(view_customer(customer_id=1))  # confirm the change actually persisted

Delete

In [141]:
def delete_customer(customer_id):
    existing = view_customer(customer_id=customer_id)
    if not existing["success"]:
        return {"success": False, "error": "Customer not found."}

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM customers WHERE customer_id = ?", (customer_id,))
        conn.commit()

    return {"success": True, "customer_id": customer_id}

In [142]:
#print(delete_customer(customer_id=3))
#print(view_customer(customer_id=3))
#Expect: deleted successfully, then "No matching customer found."

# Tools

In [143]:
save_customer_schema = {
    "type": "function",
    "function": {
        "name": "save_customer",
        "description": "Save a new customer record once all required details have been collected and confirmed with the staff member.",
        "parameters": {
            "type": "object",
            "properties": {
                "full_name": {"type": "string", "description": "Customer's full name"},
                "phone": {"type": "string", "description": "Phone number, 9-15 digits, may start with +"},
                "room_number": {"type": "string", "description": "Room number, e.g. 101 or A12"},
                "check_in": {"type": "string", "description": "Check-in date in YYYY-MM-DD format"},
                "check_out": {"type": "string", "description": "Check-out date in YYYY-MM-DD format"},
                "email": {"type": "string", "description": "Optional email address"},
                "special_requests": {"type": "string", "description": "Optional free-text special requests"}
            },
            "required": ["full_name", "phone", "room_number", "check_in", "check_out"],
            "additionalProperties": False
        }
    }
}

view_customer_schema = {
    "type": "function",
    "function": {
        "name": "view_customer",
        "description": "Look up an existing customer by their ID or by name.",
        "parameters": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "integer", "description": "Exact customer ID to look up"},
                "name": {"type": "string", "description": "Full or partial name to search for"}
            },
            "additionalProperties": False
        }
    }
}

update_customer_schema = {
    "type": "function",
    "function": {
        "name": "update_customer",
        "description": "Update one or more fields of an existing customer. The customer_id itself can never be changed.",
        "parameters": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "integer", "description": "ID of the customer to update"},
                "full_name": {"type": "string", "description": "New full name"},
                "phone": {"type": "string", "description": "New phone number"},
                "email": {"type": "string", "description": "New email address"},
                "room_number": {"type": "string", "description": "New room number"},
                "check_in": {"type": "string", "description": "New check-in date, YYYY-MM-DD"},
                "check_out": {"type": "string", "description": "New check-out date, YYYY-MM-DD"},
                "special_requests": {"type": "string", "description": "New special requests text"}
            },
            "required": ["customer_id"],
            "additionalProperties": False
        }
    }
}

delete_customer_schema = {
    "type": "function",
    "function": {
        "name": "delete_customer",
        "description": "Permanently delete a customer record. Only call this AFTER the staff member has explicitly confirmed with 'yes' in response to a confirmation question. If they say 'no', do not call this tool.",
        "parameters": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "integer", "description": "ID of the customer to delete"}
            },
            "required": ["customer_id"],
            "additionalProperties": False
        }
    }
}

list_customers_schema = {
    "type": "function",
    "function": {
        "name": "list_customers",
        "description": "List every customer currently in the database.",
        "parameters": {"type": "object", "properties": {}, "additionalProperties": False}
    }
}

search_by_check_in_date_schema = {
    "type": "function",
    "function": {
        "name": "search_by_check_in_date",
        "description": "Find all customers checking in on a specific date.",
        "parameters": {
            "type": "object",
            "properties": {
                "check_in": {"type": "string", "description": "Date in YYYY-MM-DD format"}
            },
            "required": ["check_in"],
            "additionalProperties": False
        }
    }
}

export_customers_pdf_schema = {
    "type": "function",
    "function": {
        "name": "export_customers_pdf",
        "description": "Export the full customer list as a PDF file.",
        "parameters": {"type": "object", "properties": {}, "additionalProperties": False}
    }
}




In [144]:
tools = [save_customer_schema, view_customer_schema, update_customer_schema, delete_customer_schema, 
         list_customers_schema,search_by_check_in_date_schema, export_customers_pdf_schema]

In [145]:
system_prompt = """You are a front-desk assistant for a hotel. You help staff add, view, update
and delete customer records by chatting naturally.

Rules you must always follow:
- Replies are polite, short and professional.
- When adding a new customer, ask for missing required details (full name, phone,
  room number, check-in date, check-out date) one or two at a time, not all at once.
  Email and special requests are optional - don't ask for them unless the staff member
  wants to add them.
- Never guess or invent a detail the staff member hasn't given you.
- Before deleting a customer, you must ask the staff member to confirm with a clear
  yes or no. Only call the delete_customer tool after they reply "yes". If they say
  "no", cancel the deletion and say so - do not call the tool.
- After any save, update or delete, clearly repeat back exactly what happened, so the
  staff member can check it.
- If a tool call fails (for example, invalid data or a customer not found), explain the
  problem in simple, friendly language and ask for a correction. Never say something
  succeeded if it did not.
- When looking up a customer, only show the records the staff member asked for.
- If several customers match a name search, list them briefly and ask which one is meant.
- If asked about anything unrelated to customer records (e.g. room pricing, payments,
  staff accounts), politely explain that you can only help with customer records.
- If you save a customer and a duplicate_name_warning appears, mention it to
  staff briefly after confirming the save, in case they meant to update an
  existing record instead.
- You can list all customers, search by check-in date, or export the
  customer list as a PDF when asked.
"""

In [146]:
def handle_tool_call(tool_call):
    try:
        name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        functions = {
            "save_customer": save_customer,
            "view_customer": view_customer,
            "update_customer": update_customer,
            "delete_customer": delete_customer,
            "list_customers": list_customers,
            "search_by_check_in_date": search_by_check_in_date,
            "export_customers_pdf": export_customers_pdf,
        }
        function = functions.get(name)
        if function is None:
            result = {"success": False, "error": "Unknown tool."}
        else:
            result = function(**arguments)
    except (TypeError, ValueError, sqlite3.Error) as exc:
        result = {"success": False, "error": f"Could not complete that action: {exc}"}
    except Exception:
        result = {"success": False, "error": "Something went wrong while processing that action."}

    return {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(result),
    }


def chat(message, history, pending_delete):
    pending_delete = pending_delete or {}
    reply = message.strip().lower()

    # Complete or cancel a deletion only in response to the pending confirmation.
    if pending_delete:
        if reply in {"yes", "yes please", "confirm", "confirmed"}:
            try:
                result = delete_customer(pending_delete["customer_id"])
                if result.get("success"):
                    answer = (
                        f"Done. Customer #{pending_delete['customer_id']} "
                        f"({pending_delete['full_name']}, room "
                        f"{pending_delete['room_number']}) has been deleted."
                    )
                else:
                    answer = f"I couldn't delete that customer: {result.get('error', 'Unknown error')}"
            except Exception:
                answer = "I couldn't delete the customer because of a database error."
            return answer, None

        if reply in {"no", "no thanks", "cancel", "cancel it"}:
            return "Deletion cancelled. The customer record was not changed.", None

        # A different request cancels the pending confirmation; it does not delete.
        pending_delete = None

    clean_history = []
    for item in history or []:
        if not isinstance(item, dict):
            continue
        role = item.get("role")
        content = item.get("content")
        if role in {"user", "assistant"} and isinstance(content, str):
            clean_history.append({"role": role, "content": content})

    messages = [
        {"role": "system", "content": system_prompt},
        *clean_history,
        {"role": "user", "content": message},
    ]

    try:
        while True:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=tools,
            )
            model_message = response.choices[0].message

            if response.choices[0].finish_reason != "tool_calls":
                return model_message.content or "I couldn't generate a response.", pending_delete

            messages.append({
                "role": "assistant",
                "content": model_message.content,
                "tool_calls": [
                    call.model_dump(exclude_none=True)
                    for call in model_message.tool_calls
                ],
            })

            for tool_call in model_message.tool_calls:
                if tool_call.function.name == "delete_customer":
                    arguments = json.loads(tool_call.function.arguments)
                    customer_id = arguments.get("customer_id")
                    customer = view_customer(customer_id=customer_id)

                    if not customer.get("success"):
                        tool_result = {
                            "success": False,
                            "error": "No matching customer found."
                        }
                    else:
                        record = customer["result"]
                        new_pending = {
                            "customer_id": record["customer_id"],
                            "full_name": record["full_name"],
                            "room_number": record["room_number"],
                        }
                        return (
                            f"Do you really want to delete {record['full_name']} "
                            f"(room {record['room_number']}, customer "
                            f"#{record['customer_id']})? Please reply yes or no.",
                            new_pending,
                        )
                else:
                    tool_result_message = handle_tool_call(tool_call)
                    tool_result = json.loads(tool_result_message["content"])

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result),
                })

    except Exception:
        return (
            "Sorry, I couldn't complete that request. Please check the API connection "
            "and try again.",
            pending_delete,
        )

In [147]:
def _extract_text(content):
    # Gradio 6 stores chat message content as a list of parts
    # (e.g. [{"type": "text", "text": "hii"}]) instead of a plain string.
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for part in content:
            if isinstance(part, dict) and part.get("type") == "text":
                parts.append(part.get("text", ""))
            elif isinstance(part, str):
                parts.append(part)
        return "".join(parts)
    return ""


def respond(message, history, pending_delete):
    chat_history = []

    for item in history or []:
        if isinstance(item, dict):
            role = item.get("role")
            text = _extract_text(item.get("content"))
            if role in {"user", "assistant"} and text:
                chat_history.append({"role": role, "content": text})
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            user_text, assistant_text = item
            if isinstance(user_text, str):
                chat_history.append({"role": "user", "content": user_text})
            if isinstance(assistant_text, str):
                chat_history.append({"role": "assistant", "content": assistant_text})

    answer, updated_pending = chat(message, chat_history, pending_delete)
    updated_history = chat_history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer},
    ]
    return updated_history, "", updated_pending

In [148]:
pending_delete_state = {"value": None}

def chat_fn(message, history):
    history_clean = []
    for item in history or []:
        if isinstance(item, dict):
            role = item.get("role")
            text = _extract_text(item.get("content"))
            if role in {"user", "assistant"} and text:
                history_clean.append({"role": role, "content": text})

    answer, updated_pending = chat(message, history_clean, pending_delete_state["value"])
    pending_delete_state["value"] = updated_pending
    return answer

gradio.ChatInterface(fn=chat_fn).launch(inbrowser=True, share=False)

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


# Bonus Features

List all customers

In [149]:
def list_customers():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM customers ORDER BY customer_id")
        columns = [d[0] for d in cursor.description]
        rows = cursor.fetchall()

    if not rows:
        return {"success": True, "count": 0, "results": []}

    results = [dict(zip(columns, row)) for row in rows]
    return {"success": True, "count": len(results), "results": results}

Search by check-in date

In [150]:
def search_by_check_in_date(check_in):
    valid, error = validate_customer_data(check_in=check_in)
    if not valid:
        return {"success": False, "error": error}

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM customers WHERE check_in = ?", (check_in,))
        columns = [d[0] for d in cursor.description]
        rows = cursor.fetchall()

    if not rows:
        return {"success": False, "error": f"No customers checking in on {check_in}."}

    results = [dict(zip(columns, row)) for row in rows]
    return {"success": True, "count": len(results), "results": results}

Export customer list as PDF

In [151]:
from fpdf import FPDF

def export_customers_pdf():
    data = list_customers()
    if data["count"] == 0:
        return {"success": False, "error": "No customers to export."}

    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, "Customer List", ln=True)
    pdf.set_font("Helvetica", "", 10)

    for c in data["results"]:
        line = (f"#{c['customer_id']} | {c['full_name']} | {c['phone']} | "
                f"Room {c['room_number']} | {c['check_in']} to {c['check_out']}")
        pdf.multi_cell(0, 8, line, new_x="LMARGIN", new_y="NEXT")

    path = str(Path.cwd() / "customer_list.pdf")
    pdf.output(path)
    return {"success": True, "path": path, "count": data["count"]}

Handle duplicate names

In [152]:
def save_customer(full_name, phone, room_number, check_in, check_out,
                   email=None, special_requests=None):
    valid, error = validate_customer_data(
        full_name=full_name, phone=phone, email=email,
        room_number=room_number, check_in=check_in, check_out=check_out,
        require_all=True
    )
    if not valid:
        return {"success": False, "error": error}

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT customer_id FROM customers WHERE full_name = ?", (full_name,))
        existing_matches = cursor.fetchall()

        cursor.execute("""
            INSERT INTO customers (full_name, phone, email, room_number,
                                    check_in, check_out, special_requests)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (full_name, phone, email, room_number, check_in, check_out, special_requests))
        conn.commit()
        new_id = cursor.lastrowid

    result = {"success": True, "customer_id": new_id}
    if existing_matches:
        result["duplicate_name_warning"] = (
            f"Note: {len(existing_matches)} other customer(s) already named "
            f"'{full_name}' exist (IDs: {[r[0] for r in existing_matches]})."
        )
    return result

In [153]:
print(export_customers_pdf)

<function export_customers_pdf at 0x000001EAD4F22A30>


In [154]:
from fpdf import FPDF
print("fpdf import OK")

fpdf import OK


In [155]:
result = export_customers_pdf()
print(result)

{'success': True, 'path': 'c:\\Users\\HP\\Hotel_Management_AI_Agent\\customer_list.pdf', 'count': 6}


C:\Users\HP\AppData\Local\Temp\ipykernel_8020\111907390.py:11: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, "Customer List", ln=True)


In [156]:
print([t["function"]["name"] for t in tools])

['save_customer', 'view_customer', 'update_customer', 'delete_customer', 'list_customers', 'search_by_check_in_date', 'export_customers_pdf']


In [157]:
import inspect
print("export_customers_pdf" in inspect.getsource(handle_tool_call))

True
